# Notebook 7 — April 2026: Optimal New Frappuccino Launch

**The question:** If Starbucks launches a new Frappuccino in April 2026, what should it look like to maximize sales?

**Context:** April is a spring shoulder season — warming temperatures drive the first wave of cold drink demand, but consumers haven't yet shifted fully to summer mode. CPI and wage data from FRED inform current price sensitivity.

This notebook applies the optimization framework from Notebook 4 to produce a **concrete, actionable product specification** for an April launch.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.optimize import minimize
from pathlib import Path
from IPython.display import display

np.random.seed(42)

ON_KAGGLE = Path('/kaggle/working').exists()
if ON_KAGGLE:
    DATA_DIR = Path('/kaggle/input/starbucks-recommendation-engine')
    if not DATA_DIR.exists():
        DATA_DIR = Path('/kaggle/input/datasets/shiratoriseto/starbucks-recommendation-engine')
else:
    DATA_DIR = Path('../data')

menu = pd.read_csv((DATA_DIR / 'raw' if not ON_KAGGLE else DATA_DIR) / 'starbucks_menu.csv')
fred = pd.read_csv((DATA_DIR / 'raw' if not ON_KAGGLE else DATA_DIR) / 'fred_macro.csv', parse_dates=['date'])
txn = pd.read_csv((DATA_DIR / 'processed' if not ON_KAGGLE else DATA_DIR) / 'synthetic_transactions.csv', parse_dates=['date'])

print(f'Menu: {menu.shape} | Transactions: {txn.shape} | FRED: {fred.shape}')

## Section 1 — April Market Conditions

What does April look like across our 5 cities?

In [ ]:
# ── April conditions ──────────────────────────────────────────────────
weather = pd.read_csv((DATA_DIR / 'raw' if not ON_KAGGLE else DATA_DIR) / 'weather_daily.csv', parse_dates=['date'])
april_weather = weather[weather['date'].dt.month == 4]

print('=== April Temperature by City ===')
for city in april_weather['city'].unique():
    temps = april_weather[april_weather['city'] == city]['temp_mean_f']
    print(f'  {city:15s}: mean={temps.mean():.1f}°F, range={temps.min():.0f}-{temps.max():.0f}°F')

avg_april_temp = april_weather['temp_mean_f'].mean()
print(f'\nOverall April avg: {avg_april_temp:.1f}°F')

# Latest macro
latest_cpi = fred['cpi'].iloc[-1]
latest_wage = fred['avg_hourly_earnings'].iloc[-1]
print(f'Latest CPI: {latest_cpi:.1f}')
print(f'Latest wage: ${latest_wage:.2f}/hr')

# April transaction patterns from synthetic data
april_txn = txn[txn['date'].dt.month == 4]
frapp_cats = ['Frappuccino® Blended Coffee', 'Frappuccino® Blended Crème', 'Frappuccino® Light Blended Coffee']
april_frapp_share = april_txn['category'].isin(frapp_cats).mean()
print(f'\nApril Frappuccino share: {april_frapp_share:.1%}')
print(f'April avg ticket: ${april_txn["total_price"].mean():.2f}')

## Section 2 — Optimize for April

In [ ]:
# ── Scoring function (from Notebook 4) ────────────────────────────────
frapp_menu = menu[menu['category'].isin(frapp_cats)]
IDEAL_CAL = frapp_menu['calories'].median()
IDEAL_SUGAR = frapp_menu['sugar_g'].median()
PRICE_MEAN = frapp_menu['price_usd'].mean()
PRICE_STD = frapp_menu['price_usd'].std()

def product_score(cal, sug, caf, price, temp, month, cpi, wage):
    cal_s = np.exp(-0.5 * ((cal - IDEAL_CAL) / 80) ** 2)
    sug_s = np.exp(-0.5 * ((sug - IDEAL_SUGAR) / 15) ** 2)
    caf_s = 0.7 + 0.3 * np.exp(-0.5 * ((caf - 100) / 80) ** 2)
    desirability = cal_s * 0.4 + sug_s * 0.3 + caf_s * 0.3
    
    cpi_factor = latest_cpi / cpi
    price_z = (price - PRICE_MEAN) / PRICE_STD
    affordability = 1.0 / (1.0 + np.exp(price_z * cpi_factor))
    wage_factor = wage / latest_wage
    affordability = affordability ** (1.0 / max(wage_factor, 0.5))
    
    temp_score = 1.0 / (1.0 + np.exp(-(temp - 65) / 10))
    summer_months = {5: 0.8, 6: 1.0, 7: 1.0, 8: 1.0, 9: 0.8}
    month_bonus = summer_months.get(month, 0.4)
    seasonal = temp_score * 0.6 + month_bonus * 0.4
    
    return desirability * 0.35 + affordability * 0.35 + seasonal * 0.30

# ── Optimize for each city's April temperature ────────────────────────
city_results = []
for city in april_weather['city'].unique():
    city_temp = april_weather[april_weather['city'] == city]['temp_mean_f'].mean()
    
    def neg_score(x):
        return -product_score(x[0], x[1], x[2], x[3], city_temp, 4, latest_cpi, latest_wage)
    
    bounds = [(150, 500), (20, 65), (0, 200), (4.00, 7.50)]
    best = None
    best_s = -np.inf
    for _ in range(20):
        x0 = [np.random.uniform(b[0], b[1]) for b in bounds]
        r = minimize(neg_score, x0, method='L-BFGS-B', bounds=bounds)
        if -r.fun > best_s:
            best_s = -r.fun
            best = r
    
    city_results.append({
        'city': city, 'temp_f': round(city_temp, 1),
        'calories': round(best.x[0]), 'sugar_g': round(best.x[1]),
        'caffeine_mg': round(best.x[2]), 'price': round(best.x[3], 2),
        'score': round(best_s, 4),
    })

results_df = pd.DataFrame(city_results)
display(results_df)

# National average
avg_result = results_df[['calories', 'sugar_g', 'caffeine_mg', 'price', 'score']].mean()
print(f'\n=== National Average Optimal Spec ===')
print(f'  Calories:  {avg_result["calories"]:.0f} kcal')
print(f'  Sugar:     {avg_result["sugar_g"]:.0f}g')
print(f'  Caffeine:  {avg_result["caffeine_mg"]:.0f}mg')
print(f'  Price:     ${avg_result["price"]:.2f}')
print(f'  Score:     {avg_result["score"]:.4f}')

## Section 3 — Product Specification Card

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# APRIL 2026 NEW FRAPPUCCINO — PRODUCT SPECIFICATION
# ══════════════════════════════════════════════════════════════════════

spec = results_df.iloc[results_df['score'].argmax()]  # Best-scoring city

print('=' * 60)
print('  NEW FRAPPUCCINO PRODUCT SPECIFICATION')
print('  Target Launch: April 2026 (Spring Shoulder Season)')
print('=' * 60)
print(f'''
  Product Name Suggestion: "Spring Blossom Frappuccino"
  
  ┌─────────────────────────────────────────────┐
  │  Calories:     {avg_result["calories"]:.0f} kcal                      │
  │  Sugar:        {avg_result["sugar_g"]:.0f}g                          │
  │  Caffeine:     {avg_result["caffeine_mg"]:.0f}mg                       │
  │  Grande Price: ${avg_result["price"]:.2f}                       │
  └─────────────────────────────────────────────┘
  
  Optimization Score: {avg_result["score"]:.4f}
  
  WHY THESE SPECS:
  • Calories near category median — mass appeal, not niche
  • Moderate sugar — sweet enough for treat-seekers,
    not excessive for health-conscious cross-buyers
  • Caffeine present — differentiates from Crème line,
    appeals to afternoon-energy seekers
  • Price at/below mean — April is not peak Frappuccino
    season yet, lower price captures early adopters
''')

# Compare to existing menu
print('=== vs Existing Frappuccinos ===')
for col, label in [('calories', 'Calories'), ('sugar_g', 'Sugar (g)'), ('caffeine_mg', 'Caffeine (mg)'), ('price_usd', 'Price ($)')]:
    col_key = col.replace('_usd', '')
    new_val = avg_result.get(col_key, avg_result.get('price', 0))
    existing_mean = frapp_menu[col].mean()
    diff_pct = (new_val - existing_mean) / existing_mean * 100
    arrow = '↑' if diff_pct > 0 else '↓'
    print(f'  {label:15s}: New={new_val:6.1f}  Existing={existing_mean:6.1f}  ({arrow}{abs(diff_pct):.1f}%)')

## Section 4 — Revenue Projection

In [ ]:
# ── Revenue projection for April launch ──────────────────────────────

# From synthetic data: April daily Frappuccino transactions
april_daily_frapp = april_txn[april_txn['category'].isin(frapp_cats)].groupby(
    april_txn[april_txn['category'].isin(frapp_cats)]['date'].dt.date
).size()

avg_daily_frapp = april_daily_frapp.mean()
print(f'Average daily Frappuccino transactions (April, synthetic): {avg_daily_frapp:.0f}')

# Capture scenarios
scenarios = [
    ('Conservative (10%)', 0.10),
    ('Moderate (15%)', 0.15),
    ('Optimistic (20%)', 0.20),
]

print(f'\n{"Scenario":30s} {"Daily Volume":>12s} {"Monthly Rev":>12s} {"Annual Rev*":>12s}')
print('-' * 70)
for label, rate in scenarios:
    daily_vol = avg_daily_frapp * rate
    monthly_rev = daily_vol * 30 * avg_result['price']
    annual_rev = monthly_rev * 6  # 6-month seasonal window
    print(f'{label:30s} {daily_vol:>12.0f} {"${:,.0f}".format(monthly_rev):>12s} {"${:,.0f}".format(annual_rev):>12s}')

print('\n* Annual Rev assumes 6-month availability (April-September)')
print('  Note: projections are illustrative, based on synthetic data patterns')

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
months = ['Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep']
seasonality = [0.7, 0.9, 1.0, 1.0, 0.95, 0.8]  # relative demand
base_daily = avg_daily_frapp * 0.15  # moderate scenario

for label, rate in scenarios:
    daily = avg_daily_frapp * rate
    monthly_rev = [daily * s * 30 * avg_result['price'] for s in seasonality]
    ax.plot(months, monthly_rev, 'o-', label=label, linewidth=2)

ax.set_title('Projected Monthly Revenue: New April Frappuccino', fontsize=13, fontweight='bold')
ax.set_ylabel('Monthly Revenue ($)')
ax.set_xlabel('Month')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Limitations & Next Steps

**Limitations:**
- Revenue projections are based on synthetic transaction patterns, not real POS data
- Capture rates (10-20%) are assumed, not modeled
- No competitive response or marketing spend factored in
- Flavor/taste appeal is not modeled — only nutrition and price attributes

**Next steps with real data:**
1. Calibrate capture rate from historical new product launches
2. A/B test price points ($4.99 vs $5.49 vs $5.99)
3. Regional pricing using city-level temperature sensitivity
4. Cross-sell with food items (not modeled here)